In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
import os
# =====================================================
# 1 Load dataset
# =====================================================

df = pd.read_csv("GSE124821_data_1e8.csv")

quanTIseq_cols = [
    "Dendritic_quanTIseq",
    "NK_quanTIseq",
    "T.CD8_quanTIseq",
    "Tregs_quanTIseq"
]

# =====================================================
# 2 Convert quanTIseq fractions → cell numbers
# =====================================================

scale = 1e6

for col in quanTIseq_cols:
    df[col] = df[col] * scale

# =====================================================
# 3 Estimate cancer cell number
# C = (1 - immune_fraction_sum) × 10^6
# =====================================================

df["Cancer_cells"] = (
    1 - df[quanTIseq_cols].sum(axis=1) / scale
) * scale


# =====================================================
# 4 Stratified 80/20 split
# =====================================================

df["strata"] = df["Mouse_treatment"] + "_" + df["Timepoint"]

import random

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["strata"],
    random_state=42
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))


# =====================================================
# 5 Extract average immune values
# =====================================================

def extract_averages(data):

    Cs, Ds, NKs, T8s, Tregs = [], [], [], [], []

    for day in ["day3","day7","end"]:

        day_df = data[data["Timepoint"] == day]

        Cs.append(day_df["Cancer_cells"].mean())
        Ds.append(day_df["Dendritic_quanTIseq"].mean())
        NKs.append(day_df["NK_quanTIseq"].mean())
        T8s.append(day_df["T.CD8_quanTIseq"].mean())
        Tregs.append(day_df["Tregs_quanTIseq"].mean())

    return (
        np.array(Cs),
        np.array(Ds),
        np.array(NKs),
        np.array(T8s),
        np.array(Tregs)
    )


C_train, D_train, NK_train, T8_train, Tr_train = extract_averages(train_df)
C_val, D_val, NK_val, T8_val, Tr_val = extract_averages(val_df)


# =====================================================
# 6 Initial state (from training day3)
# =====================================================

y0 = np.array([
    C_train[0],
    D_train[0],
    NK_train[0],
    Tr_train[0],
    T8_train[0]
])


# =====================================================
# 7 ODE model
# =====================================================

def ode_system(t, y, p):

    C, D, N, Tr, T8 = y

    dCdt = p["lambda_C"]*C*(1-C/p["C_M"]) - p["eta_8"]*T8*C - p["eta_N"]*N*C - p["d_C"]*C

    dDdt = p["lambda_DC_comb"]*(C/(p["K_C"] + C)) - p["d_D"]*D

    dNdt = (
        p["sigma_N"]
        - p["d_N"]*N
        - p["gamma_N"]*Tr*N
        + p["a_C"]*N*(C/(1 + C/p["beta_1"] + N/p["beta_2"]))
    )

    dTrdt = -p["d_Treg"]*Tr + p["lambda_Tr_comb"]*(C/(p["K_C"] + C))

    dT8dt = (
        -p["d_T8"]*T8
        + p["lambda_T8_comb"]*(D/(p["K_D"] + D))*(1/(1 + Tr/p["K_Treg"]))
    )

    return [dCdt, dDdt, dNdt, dTrdt, dT8dt]


# =====================================================
# 8 Base parameters
# =====================================================

base_params = {

    "lambda_C": 1.5,
    "C_M": 0.8,
    "eta_8": 328.55,
    "eta_N": 300.0,
    "d_C": 0.17,

    "lambda_DC_comb": 8e-5,
    "K_C": 0.4,
    "d_D": 0.1,

    "sigma_N": 5e-5,
    "d_N": 0.1,
    "gamma_N": 150.0,
    "a_C": 0.5,
    "beta_1": 0.4,
    "beta_2": 2e-4,

    "d_Treg": 0.2,
    "lambda_Tr_comb": 2e-4,

    "d_T8": 0.18,
    "lambda_T8_comb": 0.00108,
    "K_Treg": 2.5e-4,
    "K_D": 4e-4
}


# =====================================================
# 9 Parameters to fit
# =====================================================

fit_param_names = [
    "sigma_N",
    "lambda_T8_comb",
    "lambda_Tr_comb",
    "K_Treg",
    "lambda_DC_comb"
]

initial_params = [base_params[p] for p in fit_param_names]


# =====================================================
# 10 Model simulation
# =====================================================

def simulate_model(theta):

    params = base_params.copy()

    for i,name in enumerate(fit_param_names):
        params[name] = theta[i]

    sol = solve_ivp(
        lambda t,y: ode_system(t,y,params),
        t_span=(0,14),
        y0=y0,
        t_eval=[3,7,14],
        method="LSODA"
    )

    return sol.y


# =====================================================
# 11 Loss function
# =====================================================

def loss(theta):

    y = simulate_model(theta)

    error = (
        (y[0] - C_train)**2 +
        (y[1] - D_train)**2 +
        (y[2] - NK_train)**2 +
        (y[3] - Tr_train)**2 +
        (y[4] - T8_train)**2
    )

    return np.sum(error)


# =====================================================
# 12 Parameter optimization
# =====================================================

result = minimize(
    loss,
    initial_params,
    method="Nelder-Mead"
)

best_params = result.x


# =====================================================
# 13 Print fitted parameters
# =====================================================

print("\nBest fitted parameters:")

for i,name in enumerate(fit_param_names):
    print(name, "=", best_params[i])


# =====================================================
# 14 Validation prediction
# =====================================================

y0_val = np.array([
    C_val[0],
    D_val[0],
    NK_val[0],
    Tr_val[0],
    T8_val[0]
])

params = base_params.copy()

for i,name in enumerate(fit_param_names):
    params[name] = best_params[i]

sol = solve_ivp(
    lambda t,y: ode_system(t,y,params),
    t_span=(0,4),
    y0=y0_val,
    t_eval=[4],
    method="LSODA"
)

pred_day7 = sol.y[:, -1]


# =====================================================
# 15 Compare prediction vs real data
# =====================================================

real_day7 = [
    C_val[1],
    D_val[1],
    NK_val[1],
    Tr_val[1],
    T8_val[1]
]

predicted_day7 = pred_day7

print("\nPredicted day7:", predicted_day7)
print("Observed day7:", real_day7)

rmse = np.sqrt(np.mean((predicted_day7 - real_day7)**2))

print("\nValidation RMSE:", rmse)

Training samples: 153
Validation samples: 39

Best fitted parameters:
sigma_N = 5.25e-05
lambda_T8_comb = 0.00108
lambda_Tr_comb = 0.0002
K_Treg = 0.00025
lambda_DC_comb = 8e-05

Predicted day7: [5.42610358e-20 1.58622741e+03 2.42972208e-09 1.44047405e+02
 5.48929788e+01]
Observed day7: [np.float64(994314.3261150001), np.float64(3582.01825), np.float64(1081.46075), np.float64(546.0965000000001), np.float64(476.09838500000006)]

Validation RMSE: 444672.1198621339
